## MovieLens ALS Collaborative Filtering

This notebook builds a collaborative filtering recommendation model using **Alternating Least Squares (ALS)** on the MovieLens dataset (~3.7M ratings). The pipeline:
1. Load and preprocess ratings data (cast IDs to integers)
2. Train/test split (80/20)
3. Hyperparameter tuning via CrossValidator
4. MLflow experiment tracking
5. Evaluation (RMSE) and top-10 recommendations per user

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

SAMPLE_SIZE = 50_000

# Load tables
ratings = spark.table("frantzpaul_tech.movielens.ratings")
movies = spark.table("frantzpaul_tech.movielens.movies")

# ALS requires numeric userId and movieId
ratings_prep = (
    ratings
    .withColumn("userId", F.col("userId").cast(IntegerType()))
    .withColumn("movieId", F.col("movieId").cast(IntegerType()))
    .withColumn("rating", F.col("rating").cast(FloatType()))
    .select("userId", "movieId", "rating")
    .sample(fraction=SAMPLE_SIZE / 3_710_238 * 1.1, seed=42)
    .limit(SAMPLE_SIZE)
)

print(f"Sampled ratings: {ratings_prep.count():,}")
ratings_prep.printSchema()
display(ratings_prep.limit(5))

### Train/Test Split (80/20)

In [0]:
(train, test) = ratings_prep.randomSplit([0.8, 0.2], seed=42)

print(f"Training set: {train.count():,} ratings")
print(f"Test set:     {test.count():,} ratings")

### ALS Model Training with Hyperparameter Tuning

Using `CrossValidator` with a parameter grid over `rank`, `regParam`, and `maxIter`. The evaluator minimizes RMSE. `coldStartStrategy="drop"` handles users/items in test that are absent from training.

In [0]:
import mlflow
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import numpy as np

# Define ALS model
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42,
)

# Parameter grid (modest for 1-worker cluster)
param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [10, 50])
    .addGrid(als.regParam, [0.01, 0.1])
    .addGrid(als.maxIter, [10])
    .build()
)

# Evaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction",
)

# Cross-validator (3-fold)
cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
)

# Train with MLflow tracking
mlflow.set_experiment("/Users/frantz@frantzpaul.tech/MovieLens_ALS_Recommendation")

with mlflow.start_run(run_name="ALS_CrossValidator") as run:
    cv_model = cv.fit(train)
    best_model = cv_model.bestModel

    # Find best param combo from avgMetrics
    best_idx = int(np.argmin(cv_model.avgMetrics))
    best_params = param_grid[best_idx]
    best_rank = best_params[als.rank]
    best_reg = best_params[als.regParam]
    best_iter = best_params[als.maxIter]

    # Log best hyperparameters
    mlflow.log_param("best_rank", best_rank)
    mlflow.log_param("best_regParam", best_reg)
    mlflow.log_param("best_maxIter", best_iter)
    mlflow.log_param("coldStartStrategy", "drop")
    mlflow.log_param("nonnegative", True)
    mlflow.log_param("num_folds", 3)
    mlflow.log_param("training_rows", train.count())
    mlflow.log_param("sample_size", 50_000)

    # Evaluate on test set
    predictions = best_model.transform(test)
    rmse = evaluator.evaluate(predictions)
    mlflow.log_metric("test_rmse", rmse)

    # Log cross-validation fold metrics
    for i, avg_metric in enumerate(cv_model.avgMetrics):
        mlflow.log_metric(f"cv_avg_rmse_param_combo_{i}", avg_metric)

    # Note: To log the SparkML model artifact, set MLFLOW_DFS_TMP to a
    # UC Volume path (e.g. /Volumes/catalog/schema/volume/tmp)

    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

print(f"\nBest params: rank={best_rank}, regParam={best_reg}, maxIter={best_iter}")
print(f"Test RMSE: {rmse:.4f}")
print(f"\nAll CV results (RMSE):")
for i, (params, metric) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    print(f"  Combo {i}: rank={params[als.rank]}, regParam={params[als.regParam]} -> RMSE={metric:.4f}")

### Evaluation — Predictions vs Actuals

In [0]:
from pyspark.sql import functions as F

print(f"Test RMSE: {rmse:.4f}")
print(f"Best params: rank={best_rank}, regParam={best_reg}, maxIter={best_iter}")
print()

# Sample predictions vs actuals
sample_preds = (
    predictions
    .join(
        movies.withColumn("movieId", F.col("movieId").cast("int")),
        on="movieId",
    )
    .select("userId", "title", "rating", F.round("prediction", 2).alias("predicted"))
    .orderBy(F.rand(seed=42))
    .limit(20)
)
display(sample_preds)

### Top-10 Movie Recommendations per User

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Sample users to generate recommendations for
sample_user_ids = [1, 100, 500, 1000, 5000]
sample_users = spark.createDataFrame([(uid,) for uid in sample_user_ids], ["userId"])

# All unique movies from training data
all_movies = train.select("movieId").distinct()

# Cross-join: every sample user × every movie
user_movie_pairs = sample_users.crossJoin(all_movies)

# Predict ratings for all pairs
all_predictions = best_model.transform(user_movie_pairs).filter(F.col("prediction").isNotNull())

# Rank by predicted rating per user and keep top 10
window = Window.partitionBy("userId").orderBy(F.desc("prediction"))
top_recs = (
    all_predictions
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") <= 10)
    .join(
        movies.withColumn("movieId", F.col("movieId").cast("int")),
        on="movieId",
    )
    .select("userId", "rank", "title", "genres", F.round("prediction", 2).alias("predicted_rating"))
    .orderBy("userId", "rank")
)

print(f"Top-10 recommendations for users {sample_user_ids}:")
display(top_recs)